In [5]:
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# LOAD VÀ XỬ LÝ DỮ LIỆU
# Thay đường dẫn nếu cần thiết
gdf = gpd.read_file("D:/Dai_Hoc/HK4_dot2/Tri Tue Nhan Tao/BAI_TAP_AI/ToBanDo/gia_lai_districts.json") 
gia_lai_map = gdf[gdf['NAME_1'] == 'GiaLai'].copy()

# Thiết lập hệ tọa độ để tránh lỗi vẽ
if gia_lai_map.crs is None:
    gia_lai_map.set_crs(epsg=4326, inplace=True)
gia_lai_map = gia_lai_map.to_crs(epsg=3857)

# Xây dựng danh sách kề (Adjacency List)
adj_list = {}
for i, row in gia_lai_map.iterrows():
    neighbors = gia_lai_map[gia_lai_map.geometry.distance(row.geometry) < 0.001]['NAME_2'].tolist()
    adj_list[row['NAME_2']] = [n for n in neighbors if n != row['NAME_2']]

# KHỞI TẠO BIẾN
all_colors = ['red', 'blue', 'yellow', 'green']
domains = {district: list(all_colors) for district in gia_lai_map['NAME_2']}
assignment = {}
districts = gia_lai_map['NAME_2'].tolist()

# CÁC HÀM HỖ TRỢ
def draw_step(step_assignment):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(12, 10))
    
    current_colors = gia_lai_map['NAME_2'].map(step_assignment).fillna('lightgrey')
    gia_lai_map.plot(ax=ax, color=current_colors, edgecolor='black', linewidth=0.5)
    
    for idx, row in gia_lai_map.iterrows():
        centroid = row['geometry'].centroid
        plt.annotate(text=row['NAME_2'], xy=(centroid.x, centroid.y), 
                     ha='center', va='center', fontsize=6, fontweight='bold',
                     bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

    plt.title(f"Forward Checking: Đã gán {len(step_assignment)}/{len(districts)} huyện", fontsize=14)
    plt.axis('off')
    plt.show()
    time.sleep(0.2)

def forward_checking(district, color, current_domains):
    new_domains = {d: list(vals) for d, vals in current_domains.items()}
    for neighbor in adj_list.get(district, []):
        if neighbor not in assignment and color in new_domains[neighbor]:
            new_domains[neighbor].remove(color)
            if not new_domains[neighbor]: # Nếu hàng xóm hết màu thì nhánh này sai
                return None
    return new_domains

# THUẬT TOÁN BACKTRACKING KẾT HỢP FORWARD CHECKING
def backtrack(current_domains):
    if len(assignment) == len(districts):
        return True
    
    # Heuristic MRV: Chọn huyện có ít màu khả dụng nhất
    unassigned = [d for d in districts if d not in assignment]
    district = min(unassigned, key=lambda d: len(current_domains[d]))
    
    for color in current_domains[district]:
        old_domains = {d: list(vals) for d, vals in current_domains.items()}
        
        assignment[district] = color
        new_domains = forward_checking(district, color, current_domains)
        
        if new_domains is not None:
            draw_step(assignment)
            if backtrack(new_domains):
                return True
        
        # Quay lui
        del assignment[district]
        current_domains = old_domains
        draw_step(assignment)
            
    return False

draw_step({})
if backtrack(domains):
    print("Tô màu thành công!")
else:
    print("Không tìm được lời giải!")

AttributeError: module 'numpy' has no attribute 'matrix'